# Threshold tools

This notebook demonstrates how to use `climakitae` to evaluate the frequency of extreme weather events in downscaled climate model simulations. This type of analysis is often called "Extreme value analysis" or "1-in-X analysis".

`climakitae` uses the Annual Maxima/Minima Series (AMS) method for extreme value analysis. In the AMS method, the maximum (or minimum) variable value is obtained for each year. A statistical distribution is fitted to the values and then used to obtain return probabilities, values, or periods. This methodology is built-in to the `climakitae` `metric_calc` processor so that users can generate 1-in-X results with a single call to the `ClimateData` object. Many customizable settings are provided.

**Terms used in this notebook**:
- __Return probabilities__: The probability of a threshold being exceeded within a given period of time. 
    - For example, the maximum temperature at a location has a 10% chance of exceeding 105°F in any year.
- __Return values__: Also known as return levels. A threshold with a known exceedance probability.
    - The return value of 105°F has a return probability of 10%.
- __Return periods__: In inverse of the return probability, interpreted as the average expected wait time between threshold exceedances. 
    - If the return probability of air temperature exceeding 105°F is 10%, the return period of that event is 10 years.

**Intended Application**: As a user, I want to learn how to:
- **<span style="color:#FF0000">Generate return probabilities, return values, and return periods for climate simulations with climakitae.</span>**
- **<span style="color:#FF0000">Visualize the spatial distribution of return value results across a region.</span>**
- **<span style="color:#FF0000">Compare return periods results between different global warming levels.</span>**

**Runtime**: With the default settings, this notebook takes approximately **5 minutes** to run from start to finish. Modifications to selections may increase the runtime.

**References**: The techniques in this notebook come from applications of extreme value theory to climate data. These texts provide further reading on the topic:  
Cooley, D., 2009: Extreme value analysis and the study of climate change. Climatic Change 97, 77–83. https://doi.org/10.1007/s10584-009-9627-x  
Coles, S., 2001: An Introduction to Statistical Modeling of Extreme Values. Springer, 208 pp.  
Wilks, D. S., 2011: Statistical Methods in the Atmospheric Sciences. Academic Press, 676 pp.  


### 1-in-X options

The examples in this notebook use the `metric_calc` processor available through the `climakitae` `ClimateData` object to run 1-in-X evaluations. The following arguments can be passed to `one_in_x` in the `metric_calc` processor:

| Argument | Options | Required? | Notes |
|---|---|---|---|
| `return_periods` | List of ints | Required (or `return_values`) | Return periods in years, e.g. `[10, 25, 50]` |
| `return_values` | List of floats | Required (or `return_periods`) | Threshold values, e.g. `[105]` |
| `extremes_type` | `"max"`, `"min"` | Optional | Default: `"max"` |
| `distribution` | `"gev"`, `"genpareto"`, `"gumbel"`, `"weibull"`, `"pearson3"`, `"gamma"` | Optional | Default: `"gev"` |
| `event_duration` | `(int, "day")` | Optional | Window for sub-daily events |
| `grouped_duration` | `(int, "day")` | Optional | Consecutive-day event window for multi-day extremes |
| `block_size` | int (years) | Optional | Default: 1 |
| `goodness_of_fit_test` | bool | Optional | Default: `True`; runs KS test, returns p-values |
| `variable_preprocessing` | dict | Optional | Default: {}; 'precipitation' options are 'daily_aggregation' (bool, aggregate hourly to daily timescale) and 'remove_trace' (bool, recommended to remove spurious precipitation amounts). See Section 3 for example. |

Here are the full names of the available distributions:
- "gev": Generalized extreme value (GEV) distribution  
- "gumbel": Gumbel distribution  
- "weibull": Weibull distribution  
- "genpareto": Generalized Pareto distribution  
- "gamma": Gamma distribution  
- "pearson3": Pearson Type III distribution
  
If unsure about the correct distribution to choose, the [Generalized Extreme Value distribution](https://en.wikipedia.org/wiki/Generalized_extreme_value_distribution) ("gev") is a good starting point as it is commonly used with climate data and the AMS method (Coles 2001).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import climakitae as ck
import xarray as xr
import matplotlib.pyplot as plt
from climakitae.util.colormap import read_ae_colormap

## 1. Return values: precipitation at a single location

In this example, we calculate 1-in-X return values for daily precipitation at a single station (KSAC) across two warming levels. Here we use the bias-corrected WRF simulations, but this analysis can also be applied to the LOCA2 simulations. LOCA2 will take longer to run because of the larger number of simulations in that dataset.

In [ ]:
# Initialize the interface
cd = ck.ClimateData(verbosity=0)
return_periods = [10, 25, 50]

one_in_x_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("prec")
    .processes(
        {
            "warming_level": {"warming_levels": [0.8, 2.0]},
            "convert_units": "inches/d",
            "clip": "KSAC",
            "metric_calc": {
                "one_in_x": {
                    "return_periods": return_periods,
                    "extremes_type": "max",
                }
            },
        }
    )
    .get()
)

In [ ]:
# Removing `EC-Earth3-Veg` because it has incomplete WL 0.8
sims = [s.item() for s in one_in_x_data.sim if "EC-Earth3-Veg" not in s.item()]
one_in_x_data = one_in_x_data.sel(sim=sims)

The p-value from the Kolmogorov-Smirnov (KS) test measures how well the chosen distribution fits the annual maxima series. A p-value above 0.05 indicates an acceptable fit at the 95% confidence level. Low p-values suggest the distribution may not be appropriate for the data-- try a different `distribution` option in that case.

In [ ]:
# Examining p-values across simulation, warming_level, and one_in_x combinations
one_in_x_data.to_dataframe()[["p_values"]]

In [ ]:
# Looking at the `return_values` now that we've confirmed the p-values are all reasonable
one_in_x_data.to_dataframe()[["return_values"]].groupby(
    ["warming_level", "one_in_x"]
).mean()

**As expected, longer return periods correspond to higher return values, reflecting the fact that more severe events occur less frequently.**
<br>
<br>

# 2. Return periods: temperature over a region

In this example, we used a time-based approach to find the return period for a specific temperature threshold (115°F for a 3-day heat wave) across San Bernardino County for simulation years 2040-2060.  We then display the results as a map. 

This example demonstrates how to use the `event_duration` and `grouped_duration` keywords together to compute return periods for multi-day events, in this case a three-day event based on daily maximum temperature. 

In [ ]:
# Set the three-day heatwave temperature
tmax_3day = 115

# Reset the `ck` object to clear the previous catalog query and processes
cd.reset()

one_in_x_temp = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .experiment_id(["historical", "ssp370"])
    .institution_id("UCLA")
    .table_id("day")
    .grid_label("d02")
    .variable("t2max")
    .processes(
        {
            "time_slice": (2040, 2060),
            "convert_units": "degF",
            "clip": "San Bernardino County",
            "metric_calc": {
                "one_in_x": {
                    "return_values": [tmax_3day],
                    "block_size": 1,
                    "extremes_type": "max",
                    "event_duration": (1, "day"),
                    "grouped_duration": (3, "day"),  # select consecutive 3-day events
                }
            },
        }
    )
    .get()
)

### Visualize the return periods
Next we take the median of the return period in the five simulations and create a map.  

**Larger return periods indicate less frequent events (i.e. a longer wait time between events).** These areas have a lighter color on the map.   
**Areas with smaller return periods are more susceptible to 3-day heat waves at the selected temperature.** These are the darker colors on the map. Areas with a return period of 1 have a 100% probability of experiencing the event each year based on this analysis.

In [ ]:
# Retrieving the San Bernardino County boundary for clipping
from climakitae.new_core.data_access import DataCatalog

sb_county = DataCatalog().boundaries._ca_counties.query(
    "NAME == 'San Bernardino County'"
)

In [ ]:
# Plotting the median return periods for a 3-day, 115°F heat wave across the county
fig, ax = plt.subplots()
one_in_x_temp["return_periods"].median("sim").sel(one_in_x=tmax_3day).plot(
    vmax=40, # max color limit specific to the San Bernardino results
    vmin=1,
    cmap=read_ae_colormap("ae_orange").reversed(),
    x="lon",
    y="lat",
    ax=ax,
    cbar_kwargs={"label": "Return Period (years)"},
)
# Overlaying the county boundary
sb_county.boundary.plot(ax=ax, color="darkgrey", linewidth=1.0)
ax.set_title(
    f"Return period of a 3-day, {tmax_3day}°F heat wave\nSimulation median for 2040-2060"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 3. How Extreme Event Frequency Shifts Across Warming Levels
Another way to use these tools is to anchor on a **historically extreme precipitation threshold** and ask: *how much more frequently is this event to occur as the climate warms?*

Rather than using a single fixed threshold for all models, this example derives a **GCM-specific threshold** for each GCM: the 99.5th percentile of historical daily precipitation at KSAC. This percentile is computed only on wet days (precip > 1e-6 in/day) since we are primarily interested in the return period of these extreme precipitation events given precipitation occurs. (The low 1e-6 in/day amount is used to remove spurious trace precipitation that occurs in some simulations.) This model-specific extreme precipitation threshold is then used as the return value to estimate the model's return period at **WLs 0.8, 2.0, and 3.0**, and the results are finally combined across GCMs to get one return value per warming level.

In [ ]:
# Using the same set of GCMs as `sims` in example 1
GCMS = ["EC-Earth3", "MPI-ESM1-2-HR", "TaiESM1", "MIROC6"]

results = []

# Data retrieval and processing loop for each GCM
for gcm in GCMS:

    # Step 1: 99.5th percentile of historical max daily precip for this GCM
    hist = (
        cd.reset()
        .catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label("d02")
        .source_id(gcm)
        .experiment_id("historical")
        .variable("prec")
        .processes(
            {
                "convert_units": "inches/d",
                "clip": "KSAC",
            }
        )
        .get()
    )

    # Finding the 99.5th percentile of non-zero precipitation values, since we're interested in
    # the return period of extreme precipitation events when precipitation does occur.
    threshold = float(hist.prec.where(hist.prec > 1e-6).quantile(0.995))
    print(f"\n{gcm}: 99.5th percentile = {threshold:.3f} in/day\n")

    # Step 2: return period for that GCM-specific threshold across warming levels
    rp = (
        cd.reset()
        .catalog("cadcat")
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label("d02")
        .source_id(gcm)
        .variable("prec")
        .processes(
            {
                "warming_level": {"warming_levels": [0.8, 2.0, 3.0]},
                "convert_units": "inches/d",
                "clip": "KSAC",
                "metric_calc": {
                    "one_in_x": {
                        "return_values": [threshold],
                        "extremes_type": "max",
                        "variable_preprocessing": {
                            "precipitation": {"remove_trace": True}
                        },
                    }
                },
            }
        )
        .get()
    )
    results.append(rp.sel(one_in_x=threshold, drop=True))

return_period_data = xr.concat(results, dim="sim")

In [ ]:
# Plotting the return period for the same return value across WL 0.8 and 2.0
rp_summary = (
    return_period_data.to_dataframe()[["return_periods"]]
    .groupby("warming_level")
    .median()
)

fig, ax = plt.subplots()
rp_summary["return_periods"].plot(kind="bar", ax=ax)
ax.set_xlabel("Warming Level (°C)")
ax.set_ylabel("Return Period (years)")
ax.set_title(
    f"Return period of historical 99.5th percentile in/day precipitation event"
)
ax.set_xticks(range(len(rp_summary)))
ax.set_xticklabels([f"WL {wl}" for wl in rp_summary.index], rotation=0)
plt.tight_layout()
plt.show()

**Conclusion:** As the warming levels increase, we see that the return period for the given extreme event magnitude is decreasing. A shorter return period means that this event is occuring more frequently in these WRF simulations at higher warming levels.

<br>

## 4. Other notebooks that use threshold analyses
Now that you've seen how the different `1-in-X` calculations can be performed using `climakitae`, here's a few other notebooks that we support that use similar functionality for various other use cases:

- **[Vulnerability Assessment](../collaborative/IOU/vulnerability_assessment/vulnerability_assessment.ipynb)**: Pre-built metric tables for IOU utility planning. Uses `cava_data()` to generate standardized 1-in-X outputs, percentiles,  across multiple variables.

- **[Event Finder](../work-in-progress/event_finder.ipynb)**: Identifies specific historical or projected extreme events that match a given threshold or 1-in-X event, useful for understanding specific behavior.

- **[Custom 1-in-X](../work-in-progress/custom_1_in_x.ipynb)**: Calculates 1-in-X return values for customized metrics and/or variables.

- **[1-in-X into 8760](../collaborative/IOU/8760/one_in_x_in_8760.ipynb)**: Generates an 8760 timeseries, and inserts 1-in-X events into them, to capture what yearly behavior would look like with an extreme event.